<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Features and target timing

**[C04-M05-L01 · Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/)** · C04, SQL and Data Preparation for AI · Module 5, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** decide what a model may know when a ticket arrives, define the target `escalated_72h` exactly, keep only the tickets whose target is final, and build features from the past only.

| | |
|---|---|
| **Time** | About 50 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6. |
| **You should know** | A pandas `merge` and a `groupby`, from [Transform with Python](https://learning.nextia-ai.com/courses/sql/m04/transform-with-python/). The clean tables, from [Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/). |
| **You do not need** | The local project from Modules 1 to 4. The setup below builds the database and the clean tables here. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every script, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/) has the full explanations and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M05-L01-features-and-target-timing/features-and-target-timing-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `pandas` 3.0.6 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6
import pandas; print("pandas", pandas.__version__)

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: rebuild the earlier lessons' files

This lesson starts from files that you made in earlier lessons: the extract `data/extract/` from [Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/), the clean tables `data/clean/` from [Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/). The next cell downloads the course's finished scripts ([C04/pipeline](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C04/pipeline)) into `ticket-data` and runs them in order, so you have the same files here. It takes about 20 seconds.

You should see one line per step, ending with `clean.py: done`. On your computer, your own scripts from those lessons make the same files.

In [ ]:
# Setup: rebuild the files of the earlier lessons with the course's scripts.
STEPS = ["extract", "clean"]
for file in ["queries/extract_tickets.sql", "queries/extract_outcomes.sql"] + [f"scripts/{s}.py" for s in STEPS]:
    if not Path(file).exists():
        download(LABS + "pipeline/" + file, Path(file))
for step in STEPS:
    result = subprocess.run([sys.executable, f"scripts/{step}.py"], capture_output=True, text=True)
    if result.returncode:
        raise SystemExit(f"{step}.py failed:\n{result.stdout}{result.stderr}")
    print(f"{step}.py: done")

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One ticket, from start to end

Larkfield, an online shop for home and garden products, wants to send hard tickets to a senior agent early. When a ticket arrives, a model gives a score: how likely is it that this ticket will need a senior agent? The data is made up for this course; it has no real people.

The moment when the model must answer is the **prediction time**. Here, it is when the ticket arrives. Everything the model uses must exist at that moment.

Run the next cell. It loads the clean tables from Module 4 and shows ticket T-10033 and its outcome events.

In [ ]:
import pandas as pd

tickets = pd.read_csv("data/clean/tickets.csv", parse_dates=["created_at", "joined_on"],
                      dtype={"customer_id": "string"}, keep_default_na=False, na_values=[""])
outcomes = pd.read_csv("data/clean/outcomes.csv", parse_dates=["recorded_at"])
print(len(tickets), "clean tickets,", len(outcomes), "outcome events")

columns = ["created_at", "channel", "team", "priority", "word_count", "order_value",
           "first_reply_minutes", "priority_now", "closed_at"]
display(tickets.loc[tickets["ticket_id"] == "T-10033", columns].T)
display(outcomes[outcomes["ticket_id"] == "T-10033"])

> **Check.** You should see `1398 clean tickets, 1834 outcome events`. T-10033 was created at 08:31:54 on 9 January. It has `first_reply_minutes` 53, `priority_now` 1, and `closed_at` 22:32:02. It has two events: O-00026 `escalated` at 10:45:45, and O-00033 `resolved` (refund) at 22:32:02.

Each event in the ticket's life fills in more columns:

| Time (9 January) | What happens | Known when the ticket arrives? |
|---|---|---|
| 08:31:54 | The customer sends a web form. `created_at`, `channel`, `team`, `priority` = 3, `word_count`, `order_value` | **Yes** |
| 09:24:54 | The first reply, after 53 minutes: `first_reply_minutes` | No |
| 10:45:45 | A senior agent takes the ticket (O-00026): `priority_now` changes from 3 to 1 | No |
| 22:32:02 | The ticket is resolved (O-00033): `closed_at`, `resolution_code` | No |

The export ran months later, so it knows the future of every ticket. The model in real use does not.

- A **feature** is a column that the model uses as input.
- The **target** is the column that the model learns to predict.
- **Leakage** is information in the features that the model would not have at prediction time. It makes the evaluation score too high.

## 2. Define the target exactly

Mei, the data engineer, writes the target with Grace, the support lead:

> **`escalated_72h`** = 1 if the ticket has an `escalated` outcome event within 72 hours after `created_at`; otherwise 0.

> **Predict.** How many clean tickets have an `escalated` event? How many of them were escalated more than 72 hours after creation? Write it down, then run the cell.

In [ ]:
WINDOW = pd.Timedelta(hours=72)
first_escalation = outcomes[outcomes["status"] == "escalated"].groupby("ticket_id")["recorded_at"].min()
escalated_at = tickets["ticket_id"].map(first_escalation)
target = (escalated_at <= tickets["created_at"] + WINDOW).astype(int)

print("tickets with an escalated event:", escalated_at.notna().sum())
print("escalated more than 72 hours after creation:", (escalated_at > tickets["created_at"] + WINDOW).sum())
is_t10033 = tickets["ticket_id"] == "T-10033"
print("T-10033 escalated after:", (escalated_at - tickets["created_at"])[is_t10033].iloc[0])

> **Check.** You should see **224** tickets with an escalated event, and **30** of them escalated after more than 72 hours, so their target is 0. T-10033 escalated after `0 days 02:13:51`, so its target is 1.

`groupby(...).min()` gives the time of the **first** escalation of each ticket. A ticket with no escalation gets `NaT` ("not a time"). A comparison with `NaT` is `False`, so that ticket gets 0.

## 3. A target needs time to become final

The snapshot was taken on 2026-07-01 at 06:00. A ticket created a few hours before it had no time to escalate. If it has no escalation event yet, its target is not 0. It is **unknown**. So a ticket has a final target only if it was created at least 72 hours before the snapshot.

Run the cell. It computes the cut-off and counts the tickets created after it.

In [ ]:
SNAPSHOT = pd.Timestamp("2026-07-01 06:00:00")
cutoff = SNAPSHOT - WINDOW
too_recent = tickets["created_at"] > cutoff
print("cut-off:", cutoff)
print("too recent for a final target:", too_recent.sum())

> **Check.** You should see `cut-off: 2026-06-28 06:00:00` and **29** tickets that are too recent.

> **Your turn.** Some of the 29 recent tickets already escalated within 72 hours. Count them. Use `target` and `too_recent` from the cells above. Put the number in `already_escalated`, then run the check cell.

In [ ]:
already_escalated = 0  # replace 0 with your code

In [ ]:
if fingerprint([[int(already_escalated)]]) == "a4b8c4c188f92453":
    print("Check passed.")
else:
    print("Not yet: select the rows of target where too_recent is True, then count the values that are 1.")

The other recent tickets look like 0, but some of them may still escalate after the snapshot. Mei removes all 29 from the modelling table, and counts them. If she kept them, the newest tickets would have too many zeros, and the model would learn that recent tickets escalate less often.

## 4. What each column knows

For each column, Mei asks one question: **when is this value written?** The [lesson page](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/#what-each-column-knows) has the full table. In short:

- Written at creation, or before it (from the customer table): `channel`, `team`, `priority`, `order_value`, `word_count`, `segment`, `region`, `joined_on`. These can be features.
- Identifiers and times: `ticket_id`, `customer_id`, `created_at`. They name the row, or make features. They are not features themselves.
- Outcome events (`status`, `resolution_code`, `csat`): only for the target.

Run the cell. It lists the columns of the clean tickets table.

In [ ]:
print(list(tickets.columns))

> **Your turn.** Three columns of `tickets` are written **after** the ticket is created, so they can never be features. Put their names in the list `after_creation`, then run the check cell. Tip: one of them has almost the same name as an honest feature.

In [ ]:
after_creation = []  # write the three column names here, in quotes

In [ ]:
yours = sorted(set(after_creation))
unknown = [c for c in yours if c not in tickets.columns]
if unknown:
    print(f"Not yet: {unknown} is not a column of tickets. Copy the names from the list above.")
elif fingerprint([[c] for c in yours]) == "c189123575c9a87e":
    print("Check passed.")
else:
    print(f"Not yet: you listed {len(yours)} column(s). For each column, ask: is the value written after the ticket is created?")

Look at `priority` and `priority_now`. They have almost the same name and the same values, 1 to 3. `priority` is a fact at creation. `priority_now` is set to 1 when a ticket is escalated: it records what happened later. A column name does not tell you when a value was written. Ask the person who knows the system.

## 5. Customer history: from the past only

A customer's history can help the model. But history is a place where leakage hides easily. Customer C-0094 has six tickets with a final target. One of them, T-11014 in May, escalated.

The next cell describes C-0094 on each ticket in two ways:

- `prior_tickets_90d`: the customer's tickets created in the 90 days **before** this ticket.
- `rate_all_tickets`: the customer's escalation rate over **all** their tickets.

> **Predict.** Which of the two columns was known on 14 January, when T-10072 arrived? Then run the cell.

In [ ]:
c94 = tickets[(tickets["customer_id"] == "C-0094") & ~too_recent].sort_values("created_at")
c94 = c94[["ticket_id", "created_at"]].assign(escalated_72h=target)
c94["prior_tickets_90d"] = [
    ((c94["created_at"] >= t - pd.Timedelta(days=90)) & (c94["created_at"] < t)).sum()
    for t in c94["created_at"]
]
c94["rate_all_tickets"] = round(c94["escalated_72h"].mean(), 2)
c94

> **Check.** You should see 6 rows. `prior_tickets_90d` is 0, 1, 2, 2, 2, 2. `rate_all_tickets` is 0.17 on every row: 1 escalated ticket of 6.

On 14 January, nobody knew that C-0094 would write six tickets, or that one would escalate in May. The value 0.17 on T-10072 contains the future. Worse, the rate contains the target of T-11014 itself. `prior_tickets_90d` uses only earlier tickets, so each value was known when its ticket arrived. In Lesson 3, you measure how much a leaky rate inflates a score.

## 6. Worked example: `build_features.py`

Mei puts these decisions into one script. It reads the clean files, makes the target, removes the tickets that are too recent and adds the features. The [lesson page](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/#worked-example-build_featurespy) explains each important line.

Run the next cell. `%%writefile` saves the cell as `scripts/build_features.py`, as you do in your editor. It prints `Writing scripts/build_features.py` (or `Overwriting` the second time).

In [ ]:
%%writefile scripts/build_features.py
"""Build the modelling table: one row per ticket, with only the facts known
when the ticket is created, and the target, escalated_72h."""

import json
from pathlib import Path

import pandas as pd

IN = Path("data/clean")
OUT = Path("data/model")
SNAPSHOT = pd.Timestamp("2026-07-01 06:00:00")  # from data/extract/manifest.json
WINDOW = pd.Timedelta(hours=72)
TIME_FORMAT = "%Y-%m-%d %H:%M:%S"

# Columns filled in after the ticket is created. They describe the future of
# the ticket, so they must not be features.
AFTER_CREATION = ["priority_now", "first_reply_minutes", "closed_at"]
FEATURES = ["channel", "team", "priority", "segment", "region", "order_value", "word_count",
            "customer_tenure_days", "prior_tickets_90d", "created_hour"]


def main():
    tickets = pd.read_csv(IN / "tickets.csv", parse_dates=["created_at", "joined_on"],
                          dtype={"customer_id": "string"}, keep_default_na=False, na_values=[""])
    outcomes = pd.read_csv(IN / "outcomes.csv", parse_dates=["recorded_at"])

    # Target: an escalation within 72 hours of creation.
    first_escalation = outcomes[outcomes["status"] == "escalated"].groupby("ticket_id")["recorded_at"].min()
    tickets["escalated_at"] = tickets["ticket_id"].map(first_escalation)
    tickets["escalated_72h"] = (tickets["escalated_at"] <= tickets["created_at"] + WINDOW).astype(int)

    # A ticket's 72 hours must be over before the snapshot, or its label is
    # not final yet.
    eligible = tickets["created_at"] <= SNAPSHOT - WINDOW
    counts = {"clean_tickets": len(tickets), "too_recent_for_label": int((~eligible).sum())}
    df = tickets[eligible].copy()

    # Features known at creation time.
    df["customer_tenure_days"] = (df["created_at"] - df["joined_on"]).dt.days
    df["created_hour"] = df["created_at"].dt.hour
    df = df.sort_values(["created_at", "ticket_id"])
    df["prior_tickets_90d"] = prior_count(df, tickets)

    columns = ["ticket_id", "customer_id", "created_at"] + FEATURES + ["escalated_72h"]
    df = df[columns]
    OUT.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT / "tickets_model.csv", index=False, date_format=TIME_FORMAT)
    counts.update({"rows": len(df), "escalated": int(df["escalated_72h"].sum()),
                   "escalated_rate": round(df["escalated_72h"].mean(), 3)})
    (OUT / "build_summary.json").write_text(json.dumps(counts, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(counts, indent=2))


def prior_count(df, all_tickets):
    """For each ticket: how many tickets the same customer created in the 90
    days before it. Only earlier tickets count, so the value is known at
    creation time. Guests have no history: 0."""
    history = all_tickets.dropna(subset=["customer_id"]).groupby("customer_id")["created_at"].apply(sorted)
    result = []
    for customer, created in zip(df["customer_id"], df["created_at"]):
        if pd.isna(customer):
            result.append(0)
            continue
        earlier = [t for t in history[customer] if created - pd.Timedelta(days=90) <= t < created]
        result.append(len(earlier))
    return result


if __name__ == "__main__":
    main()

Run the script. `%run` runs a file, like `python scripts/build_features.py` in a terminal.

> **Predict.** 1,398 clean tickets, 29 too recent. How many rows will the modelling table have?

In [ ]:
%run scripts/build_features.py

> **Check.** You should see `"clean_tickets": 1398`, `"too_recent_for_label": 29`, `"rows": 1369`, `"escalated": 188` and `"escalated_rate": 0.137`.

Count every row: 1,398 clean tickets − 29 too recent = **1,369 rows**. 188 of them escalated within 72 hours: 188 ÷ 1,369 = **13.7%**. The same numbers are saved in `data/model/build_summary.json`.

## 7. Guided practice: check the table

Look at one row. `.T` turns the row into a column, so it is easy to read. Run the cell.

In [ ]:
model = pd.read_csv("data/model/tickets_model.csv")
print(model.shape)
model[model["ticket_id"] == "T-11241"].T

> **Check.** You should see `(1369, 14)`: three identifiers and times, 10 features and the target. T-11241 has `customer_id` C-0094, `prior_tickets_90d` 2, `customer_tenure_days` 2506.0 and `escalated_72h` 0. `priority_now`, `first_reply_minutes` and `closed_at` are not in the table.

Now check `prior_tickets_90d` = 2 with SQL, in a way that does not use pandas. Run the next cell. It saves the query as `queries/50-prior-tickets.sql`.

In [ ]:
%%writefile queries/50-prior-tickets.sql
SELECT ticket_id, created_at
FROM tickets
WHERE customer_id = 'C-0094'
  AND created_at >= datetime('2026-06-11 17:53:05', '-90 days')
  AND created_at < '2026-06-11 17:53:05'
ORDER BY created_at;

Run the query on `larkfield.db`, the raw export. `sql()` works like `python q.py queries/50-prior-tickets.sql`.

In [ ]:
sql(Path("queries/50-prior-tickets.sql").read_text())

> **Check.** You should see 2 rows: T-11014 (2026-05-13) and T-11085 (2026-05-22). That is the same as the script. The query runs on the raw export. That is fine for C-0094, who has no duplicate rows.

`datetime('2026-06-11 17:53:05', '-90 days')` is the start of the 90 days. `>=` includes a ticket at that exact second. `<` leaves out the ticket itself: with `<=`, T-11241 counts itself, and the value becomes 3.

> **Your turn.** Change the query to list the earlier tickets of C-0094 in the 90 days before **T-11014**, created at `2026-05-13 11:25:10`. Change both times. Then run the check cell. It must match the value 2 of T-11014 in the table above.

In [ ]:
prior_t11014 = """
SELECT ticket_id, created_at
FROM tickets
WHERE customer_id = 'C-0094'
  AND created_at >= datetime('2026-06-11 17:53:05', '-90 days')
  AND created_at < '2026-06-11 17:53:05'
ORDER BY created_at;
"""
sql(prior_t11014)

In [ ]:
check(prior_t11014, rows=2, columns=2, expected="d35e456fa26bf76d", ordered=True)

## 8. Your turn: a 48-hour target

Omar, the product manager, asks: "What if we route tickets that escalate within 48 hours instead?" Change one thing and look at the result.

> **Predict.** With 48 hours, will more or fewer tickets be too recent? Will the escalated count go up or down? Write it down.

> **Your turn.** In the next cell, make a copy of the script with two changes:
>
> 1. Change `WINDOW = pd.Timedelta(hours=72)` to 48 hours.
> 2. Change the output names to `tickets_model_48h.csv` and `build_summary_48h.json`. The rest of the module needs the 72-hour files.
>
> Use `script.replace(old, new)` once for each change. Then run the cell and the check cell.

In [ ]:
script = Path("scripts/build_features.py").read_text()
script_48h = script  # add .replace(...) for each change
Path("scripts/build_features_48h.py").write_text(script_48h)
%run scripts/build_features_48h.py

In [ ]:
import json
summary_file = Path("data/model/build_summary_48h.json")
if not summary_file.exists():
    print("Not yet: data/model/build_summary_48h.json does not exist. Change the output names in the copy.")
else:
    s = json.loads(summary_file.read_text())
    values = [s["clean_tickets"], s["too_recent_for_label"], s["rows"], s["escalated"], s["escalated_rate"]]
    if s["too_recent_for_label"] == 29:
        print("Not yet: 29 tickets are still too recent, as with 72 hours. Change the window to 48 hours.")
    elif fingerprint([values]) == "e344c9aa3e538c3c":
        print("Check passed.")
    else:
        print("Not yet: the numbers are different. Change only the window and the two output names.")

The cut-off moves to 2026-06-29 06:00, so **fewer** tickets are too recent: 22 instead of 29. The table has 7 more rows. Some escalations between 48 and 72 hours now count as 0, but the new recent rows add escalations. So the escalated count goes **up**, from 188 to 190. The two changes push in opposite directions; only the run tells you the result.

Run the cell. It deletes the copy and its two files, so they do not confuse the next lesson.

In [ ]:
for file in ["scripts/build_features_48h.py", "data/model/tickets_model_48h.csv", "data/model/build_summary_48h.json"]:
    Path(file).unlink(missing_ok=True)
print(sorted(p.name for p in Path("data/model").iterdir()))

> **Check.** You should see `['build_summary.json', 'tickets_model.csv']`.

## 9. Failure case: a feature that works too well

Tomás, the AI engineer, says: "Use `priority_now`. It works so well." Run the cell. It joins `priority_now` back from the clean file and counts the targets for each value, then does the same for `priority`.

In [ ]:
clean = pd.read_csv("data/clean/tickets.csv")
model = pd.read_csv("data/model/tickets_model.csv")
both = model.merge(clean[["ticket_id", "priority_now"]], on="ticket_id", validate="one_to_one")
print(pd.crosstab(both["priority_now"], both["escalated_72h"]))
print(pd.crosstab(both["priority"], both["escalated_72h"]))

> **Check.** You should see that all 188 escalated tickets have `priority_now` = 1, and no ticket with `priority_now` 2 or 3 escalated. With `priority`, escalated tickets appear at every value: 64, 67 and 57.

**Cause:** `priority_now` is written **after** the escalation. It does not predict the escalation; it records it. At prediction time, `priority_now` is equal to `priority`, and the strong pattern is not there.

**Fix:** ask when each value is written, and keep the answer in the code. `build_features.py` lists the column in `AFTER_CREATION` and leaves it out of the output. A feature that separates the classes almost perfectly is a reason to check for leakage, not a reason to celebrate.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you learned that the prediction time decides which columns can be features. The target `escalated_72h` is exact, and the 29 tickets without a final target are removed and counted. `priority_now`, `first_reply_minutes` and `closed_at` are never features. Customer history comes from earlier tickets only. `build_features.py` writes 1,369 rows, 188 escalated (13.7%), with 10 features.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Choose a split](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.